# MAG-S3c в Google Colab

Среда выполнения: **TPU v6e-1** (44 ядра, OpenBLAS SkylakeX), как прогоны MAG-S3 и MAG-S3b. Если её нет, подойдёт
v5e с 24 ядрами; сверка окружения в ячейке 3 запишет разницу в последних знаках.

1. Ячейки 1 и 2 готовят среду: файлы со сверкой SHA-256, пакеты закреплённых версий, Google Диск.
2. Ячейка 3 гонит тесты, повторяет калибровку C1–C4 и сравнивает её число за числом с эталонной калибровкой.
3. Ячейка 4 гонит слепую проверку конвейера на подменённых скоростях и удаляет выход непросмотренным.
4. Ячейка 5 – подтверждающий прогон. Она ждёт ответа «запускай», который вводится вручную после публикации SHA-256,
   и без него не начинается.

In [ ]:
# 1. Файлы MAG-S3c и их SHA-256
import base64, hashlib, io, json, os, subprocess, sys, tarfile, time
ROOT, WORK, DRIVE = "/content/mag_s3c", "/content/work", "/content/drive/MyDrive/MAG-S3c"
SHA256 = {
 "analyze.py": "ecc6a15b77e4f4d1c07a4217439e0c97868b9a12971c9ada255f79840ef1d21f",
 "analyze_s3c.py": "3b6765b96433ee9cb1fd9e996d5573626092d0307412f90efc57ebfec56e4e36",
 "data/6PU0.pdb": "39b25015964ecd1f53a09391c7ccd1515c31780c96a9e8861c4346f68e445a3b",
 "mag_s2.py": "2352aefe0fe5974732c318b365f4fb5477ffc47fb78b12748bca2515465829c4",
 "mag_s3.py": "1df703af63a2af71a94385d33276ad39a422e9abaa040fdf4bcb618344e1f350",
 "mag_s3b.py": "f88118db34507156d3204c00368ba14c88c884e45f17ada8855da1fc45cd0ee3",
 "mag_s3c.py": "fd337b3abe932596e15db47d4de86ad54df9a9a4077b90262ffba435ffa84247",
 "physics.json": "647bc0ccb1312991a1b772193cac39f4ced1481c591422c50659bb822a3100c5",
 "reference_calibration/calibration.json": "da58950405fe340e11c1d17fab8aa2fcd7169674dd8e87d9cb1600635d302ecd",
 "requirements.txt": "ba5ddab58d78c37fbd70099684586d8e7a59b92d99d810e8768608c7f721f3a2",
 "scripts/env_check.py": "b2224d547175745ff048c6afa488cf9a681d530b4f1bf769481b022b76f65243",
 "scripts/pipeline.sh": "f65491a8816669d8b0aad30f933906d1f58f80b96bcf3bc48039eaed7040c243",
 "test_s3c.py": "48d4eabc90ab9488caf669d51a15197416206fc30f2ecf4823ef3d8f1db921f6"
}
CODE_ID = hashlib.sha256(json.dumps(SHA256, sort_keys=True).encode()).hexdigest()
assert CODE_ID == "9d737468208bc89be0d3c2d8cbe9ccdb49de5b200ff80fe0e593cf182a0c02c0"
PAYLOAD = """
...
"""
os.makedirs(ROOT, exist_ok=True)
os.makedirs(WORK, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD)), mode="r:gz") as tar:
    tar.extractall(ROOT, filter="data")


def bad_files():
    return [name for name, h in SHA256.items()
            if hashlib.sha256(open(os.path.join(ROOT, name), "rb").read()).hexdigest() != h]


def marker_ok(path):
    """A marker of cells 3 and 4 counts only when it was written for this file set."""
    return os.path.exists(path) and json.load(open(path)).get("code_id") == CODE_ID


def write_marker(path):
    with open(path, "w") as fh:
        json.dump(dict(code_id=CODE_ID, time_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())), fh)


bad = bad_files()
print("все SHA-256 совпадают с вшитыми" if not bad else f"РАСХОЖДЕНИЕ: {bad}", "; сборка", CODE_ID[:16])
assert not bad
mem = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2 ** 30
print(f"ядер: {os.cpu_count()}, память: {mem:.0f} ГБ, Python {sys.version.split()[0]}")


def sh(cmd, cwd=ROOT):
    env = dict(os.environ, WORKERS=str(os.cpu_count()))
    p = subprocess.Popen(["bash", "-c", cmd], cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

In [ ]:
# 2. Пакеты закреплённых версий и Google Диск
rc = sh("set -o pipefail; pip install -q -r requirements.txt 2>&1 | tail -3")
assert rc == 0, "pip install failed"
sh("python3 -I -c 'import sys, numpy, scipy, radicalpy; print(sys.version.split()[0], numpy.__version__, "
   "scipy.__version__, radicalpy.__version__)'")
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE, exist_ok=True)
print("Диск подключён:", DRIVE)

In [ ]:
# 3. Тесты, повтор калибровки C1-C4 и сверка окружения с эталоном (несколько минут на 44 ядрах)
t0 = time.time()
cal = f"{WORK}/calibration"
sh(f"rm -rf {cal}")
os.makedirs(cal, exist_ok=True)
rc = sh(f"set -o pipefail; python3 -I test_s3c.py --fast 2>&1 | tee {cal}/tests.log")
rc = rc or sh(f"python3 -I mag_s3c.py calibrate --out {cal} --workers {os.cpu_count()}")
rc = rc or sh(f"python3 -I scripts/env_check.py reference_calibration {cal} | tee {cal}/env_check.txt; "
              "exit ${PIPESTATUS[0]}")
if rc == 0:
    write_marker(f"{cal}/CELL3_OK")
sh(f"rm -f {cal}/C1_yields.npz; rm -rf {DRIVE}/calibration && cp -r {cal} {DRIVE}/calibration")
print(f"\nкод выхода {rc}, {(time.time() - t0) / 60:.1f} мин; калибровка и сверка окружения: {DRIVE}/calibration")

In [ ]:
# 4. Слепая проверка конвейера на подменённых скоростях; печатает только технические факты и удаляет выход
t0 = time.time()
if not marker_ok(f"{WORK}/calibration/CELL3_OK"):
    sh(f"rm -rf {WORK}/calibration && cp -r {DRIVE}/calibration {WORK}/calibration")
assert marker_ok(f"{WORK}/calibration/CELL3_OK"), "ячейка 3 не прошла на этой сборке: тесты, калибровка или сверка"
sh(f"rm -f {DRIVE}/BLIND_OK")
rc = sh(f"set -o pipefail; bash scripts/pipeline.sh blind {WORK} 1 2>&1 | tee -a {DRIVE}/blind_check.log")
if rc == 0:
    write_marker(f"{DRIVE}/BLIND_OK")
print(f"\nкод выхода {rc} (3 – пауза по времени, запусти ячейку снова), {(time.time() - t0) / 60:.1f} мин; "
      f"журнал: {DRIVE}/blind_check.log")

Ячейка 5 – подтверждающий прогон. Сначала она сверяет файлы с вшитыми SHA-256 и отметку слепой проверки этой
сборки, потом спрашивает подтверждение: ответ «запускай» вводится вручную после заморозки и публикации SHA-256 в
Telegram, а любой другой ответ останавливает ячейку до начала счёта. Время ответа записывается в
`MAG-S3c/confirmation.txt` на Диске. Прогон идёт частями по полчаса и после каждой части кладёт архив
`MAG-S3c/confirmatory_s3c.tar.gz` на Диск; после обрыва сессии достаточно выполнить ячейки 1 и 2 и снова запустить
эту ячейку с тем же ответом.

In [ ]:
# 5. Подтверждающий прогон MAG-S3c, только после ответа «запускай»
assert not bad_files(), "файлы отличаются от вшитых SHA-256"
assert marker_ok(f"{DRIVE}/BLIND_OK"), "слепая проверка ячейки 4 не прошла на этой сборке или не запускалась"
answer = input("SHA-256 опубликованы? Для старта введи «запускай»: ").strip().strip("«»\"'")
if answer != "запускай":
    raise SystemExit("подтверждения нет, прогон не начат")
t0 = time.time()
arch = f"{DRIVE}/confirmatory_s3c.tar.gz"
if os.path.exists(arch) and not os.path.exists(f"{WORK}/confirmatory_s3c"):
    sh(f"tar -xzf {arch} -C {WORK}")
stamp = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
with open(f"{WORK}/confirmation.json", "w") as fh:
    json.dump(dict(answer=answer, time_utc=stamp, code_id=CODE_ID, sha256=SHA256), fh, ensure_ascii=False, indent=1)
with open(f"{DRIVE}/confirmation.txt", "a") as fh:
    fh.write(f"{stamp} confirmation «{answer}», build {CODE_ID}\n")
print("подтверждение принято", stamp)
while True:
    rc = sh(f"set -o pipefail; bash scripts/pipeline.sh confirmatory {WORK} 0.5 2>&1 | tee -a {WORK}/confirmatory.log")
    sh(f"tar -czf {WORK}/part.tar.gz -C {WORK} confirmatory_s3c confirmation.json && cp {WORK}/part.tar.gz {arch}.tmp && "
       f"mv {arch}.tmp {arch}")
    if rc != 3:
        break
sh(f"cp {WORK}/confirmatory.log {DRIVE}/; sha256sum {arch}")
print(f"\nкод выхода {rc}, {(time.time() - t0) / 60:.1f} мин; архив: {arch}")